In [ ]:
# @title 1.Prepare Environment
!pip install torch==2.6.0 torchvision==0.21.0
%cd /content

from IPython.display import clear_output

!git clone --branch ComfyUI_v0.3.36 https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
clear_output()

%cd /content
!git clone https://github.com/Isi-dev/Practical-RIFE
%cd /content/Practical-RIFE
!pip install git+https://github.com/rk-exxec/scikit-video.git@numpy_deprecation
!mkdir -p /content/Practical-RIFE/train_log
clear_output()

%cd /content/ComfyUI

import subprocess
import sys


def install_pip_packages():
    packages = [
        'torchsde',
        'av',
        'diffusers',
        'transformers',
        'xformers==0.0.29.post2',
        'accelerate',
        # 'omegaconf',
        'tqdm',
        # 'librosa',
        # 'triton',
        # 'sageattention',
        'color-matcher',
        'einops'
    ]

    for package in packages:
        try:
            # Run pip install silently (using -q)
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '-q', package],
                check=True,
                capture_output=True
            )
            print(f"✓ {package} installed")
        except subprocess.CalledProcessError as e:
            print(f"✗ Error installing {package}: {e.stderr.decode().strip() or 'Unknown error'}")

def install_apt_packages():
    packages = ['aria2']

    try:
        # Run apt install silently (using -qq)
        subprocess.run(
            ['apt-get', '-y', 'install', '-qq'] + packages,
            check=True,
            capture_output=True
        )
        print("✓ apt packages installed")
    except subprocess.CalledProcessError as e:
        print(f"✗ Error installing apt packages: {e.stderr.decode().strip() or 'Unknown error'}")

# Run installations
print("Installing pip packages...")
install_pip_packages()
clear_output()  # Clear the pip installation output

print("Installing apt packages...")
install_apt_packages()
clear_output()  # Clear the apt installation output

print("Installation completed with status:")
print("- All pip packages installed successfully" if '✗' not in install_pip_packages.__code__.co_consts else "- Some pip packages had issues")
print("- apt packages installed successfully" if '✗' not in install_apt_packages.__code__.co_consts else "- apt packages had issues")


import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import imageio
import subprocess
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
import glob
from IPython.display import Video as outVid
import datetime
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode,
    VAELoader,
    KSampler,
    UNETLoader,
    LoraLoaderModelOnly,
    ImageScale,
    LoadImage
    # CLIPVisionLoader,
    # CLIPVisionEncode
)

from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM
from comfy_extras.nodes_wan import WanVaceToVideo
from comfy_extras.nodes_wan import TrimVideoLatent

from pathlib import Path

def download_with_aria2c(link, folder="/content/ComfyUI/models/loras"):
    import os

    filename = link.split("/")[-1]
    command = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {link} -d {folder} -o {filename}"

    print("Executing download command:")
    print(command)

    os.makedirs(folder, exist_ok=True)
    get_ipython().system(command)

    return filename

def download_civitai_model(civitai_link, civitai_token, folder="/content/ComfyUI/models/loras"):
    import os
    import time

    os.makedirs(folder, exist_ok=True)

    try:
        model_id = civitai_link.split("/models/")[1].split("?")[0]
    except IndexError:
        raise ValueError("Invalid Civitai URL format. Please use a link like: https://civitai.com/api/download/models/1523247?...")

    civitai_url = f"https://civitai.com/api/download/models/{model_id}?type=Model&format=SafeTensor"
    if civitai_token:
        civitai_url += f"&token={civitai_token}"

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"model_{timestamp}.safetensors"

    full_path = os.path.join(folder, filename)

    download_command = f"wget --max-redirect=10 --show-progress \"{civitai_url}\" -O \"{full_path}\""
    print("Downloading from Civitai...")

    os.system(download_command)

    local_path = os.path.join(folder, filename)
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        print(f"LoRA downloaded successfully: {local_path}")
    else:
        print(f"❌ LoRA download failed or file is empty: {local_path}")

    return filename


def download_lora(link, folder="/content/ComfyUI/models/loras", civitai_token=None):
    """
    Download a model file, automatically detecting if it's a Civitai link or huggingface download.

    Args:
        link: The download URL (either huggingface or Civitai)
        folder: Destination folder for the download
        civitai_token: Optional token for Civitai downloads (required if link is from Civitai)

    Returns:
        The filename of the downloaded model
    """
    if "civitai.com" in link.lower():
        if not civitai_token:
            raise ValueError("Civitai token is required for Civitai downloads")
        return download_civitai_model(link, civitai_token, folder)
    else:
        return download_with_aria2c(link, folder)



def model_download(url: str, dest_dir: str, filename: str = None, silent: bool = True) -> bool:
    """
    Colab-optimized download with aria2c

    Args:
        url: Download URL
        dest_dir: Target directory (will be created if needed)
        filename: Optional output filename (defaults to URL filename)
        silent: If True, suppresses all output (except errors)

    Returns:
        bool: True if successful, False if failed
    """
    try:
        # Create destination directory
        Path(dest_dir).mkdir(parents=True, exist_ok=True)

        # Set filename if not specified
        if filename is None:
            filename = url.split('/')[-1].split('?')[0]  # Remove URL parameters

        # Build command
        cmd = [
            'aria2c',
            '--console-log-level=error',
            '-c', '-x', '16', '-s', '16', '-k', '1M',
            '-d', dest_dir,
            '-o', filename,
            url
        ]

        # Add silent flags if requested
        if silent:
            cmd.extend(['--summary-interval=0', '--quiet'])
            print(f"Downloading {filename}...", end=' ', flush=True)

        # Run download
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)

        if silent:
            print("Done!")
        else:
            print(f"Downloaded {filename} to {dest_dir}")
        return filename

    except subprocess.CalledProcessError as e:
        error = e.stderr.strip() or "Unknown error"
        print(f"\nError downloading {filename}: {error}")
        return False
    except Exception as e:
        print(f"\nError: {str(e)}")
        return False


model_quant = "Q8_0"
#["Q4_K_M", "Q5_0", "Q5_K_M", "Q6_K", "Q8_0"]

#NOT REQUIRED
#===================================================
download_loRA_1 = False
lora_1_download_url = "https://civitai.com/api/download/models/1776890?type=Model&format=SafeTensor"

download_loRA_2 = False
lora_2_download_url = "Put your loRA here"

token_if_civitai_url = "54ffe21f4b49cb45784c5f8818929938"
#===================================================

lora_1 = None
if download_loRA_1:
    lora_1 = download_lora(lora_1_download_url, civitai_token=token_if_civitai_url)
# Validate loRA file extension
valid_extensions = {'.safetensors', '.ckpt', '.pt', '.pth', '.sft'}
if lora_1:
    if not any(lora_1.lower().endswith(ext) for ext in valid_extensions):
        print(f"❌ Invalid LoRA format: {lora_1}")
        lora_1 = None
    else:
        clear_output()
        print("loRA 1 downloaded succesfully!")

lora_2 = None
if download_loRA_2:
    lora_2 = download_lora(lora_2_download_url, civitai_token=token_if_civitai_url)
if lora_2:
    if not any(lora_2.lower().endswith(ext) for ext in valid_extensions):
        print(f"❌ Invalid LoRA format: {lora_2}")
        lora_2 = None
    else:
        clear_output()
        print("loRA 2 downloaded succesfully!")


# !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors -d /content/ComfyUI/models/clip_vision -o clip_vision_h.safetensors

# model_quant = "Q8_0"

if model_quant == "Q4_K_M":
    dit_model = model_download("https://huggingface.co/QuantStack/Wan2.1-VACE-14B-GGUF/resolve/main/Wan2.1-VACE-14B-Q4_K_M.gguf", "/content/ComfyUI/models/diffusion_models")
elif model_quant == "Q5_0":
    dit_model = model_download("https://huggingface.co/QuantStack/Wan2.1-VACE-14B-GGUF/resolve/main/Wan2.1-VACE-14B-Q5_0.gguf", "/content/ComfyUI/models/diffusion_models")
elif model_quant == "Q5_K_M":
    dit_model = model_download("https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/Wan2.1-VACE-14B-Q5_K_M.gguf", "/content/ComfyUI/models/diffusion_models")
elif model_quant == "Q6_K":
    dit_model = model_download("https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/Wan2.1-VACE-14B-Q6_K.gguf", "/content/ComfyUI/models/diffusion_models")
else:
    dit_model = model_download("https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/Wan2.1-VACE-14B-Q8_0.gguf", "/content/ComfyUI/models/diffusion_models")
causvid_lora = model_download("https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/Wan21_CausVid_14B_T2V_lora_rank32.safetensors", "/content/ComfyUI/models/loras")
# causvid_lora = model_download("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan21_CausVid_14B_T2V_lora_rank32_v2.safetensors", "/content/ComfyUI/models/loras")
text_encoder = model_download("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "/content/ComfyUI/models/text_encoders")
vae_model = model_download("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors", "/content/ComfyUI/models/vae")


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()

def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"

    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webp(images, filename_prefix, fps, quality=90, lossless=False, method=4, output_dir="/content/ComfyUI/output"):
    """Save images as animated WEBP using imageio."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webp"


    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]


    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'lossless': bool(lossless),
        'method': int(method)
    }

    with imageio.get_writer(
        output_path,
        format='WEBP',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=32, output_dir="/content/ComfyUI/output"):
    """Save images as WEBM using imageio."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"


    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]


    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'codec': str(codec),
        'output_params': ['-crf', str(int(quality))]
    }

    with imageio.get_writer(
        output_path,
        format='FFMPEG',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    """Save single frame as PNG image."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"

    frame = (image.cpu().numpy() * 255).astype(np.uint8)

    Image.fromarray(frame).save(output_path)

    return output_path


def upload_image():
    """Handle image upload in Colab and store in /content/ComfyUI/input/"""
    from google.colab import files
    import os
    import shutil

    os.makedirs('/content/ComfyUI/input', exist_ok=True)

    uploaded = files.upload()

    # Move each uploaded file to ComfyUI input directory
    for filename in uploaded.keys():
        src_path = f'/content/ComfyUI/{filename}'
        dest_path = f'/content/ComfyUI/input/{filename}'

        shutil.move(src_path, dest_path)
        print(f"Image saved to: {dest_path}")
        return dest_path

    return None

vid_15fps = ""

def colormatch(image_ref, image_target, method='mkl', strength=1.0):
    try:
        from color_matcher import ColorMatcher
    except ImportError:
        raise Exception("Can't import color-matcher")

    cm = ColorMatcher()
    image_ref, image_target = image_ref.cpu(), image_target.cpu()

    ref_np = image_ref.squeeze().numpy()
    target_np = image_target.squeeze().numpy()
    batch_size = image_target.size(0)

    if image_ref.size(0) > 1 and image_ref.size(0) != batch_size:
        raise ValueError("ColorMatch: Use either a single reference image or a batch matching target size.")

    out = []
    for i in range(batch_size):
        tgt = target_np if batch_size == 1 else target_np[i]
        ref = ref_np if image_ref.size(0) == 1 else ref_np[i]
        try:
            matched = cm.transfer(src=tgt, ref=ref, method=method)
            result = tgt + strength * (matched - tgt)
            out.append(torch.from_numpy(result))
        except Exception as e:
            print(f"Color match error: {e}")
            break

    return (torch.stack(out).float().clamp(0, 1),)

def image_width_height(image):
    if image.ndim == 4:
        _, height, width, _ = image.shape
    elif image.ndim == 3:
        height, width, _ = image.shape
    else:
        raise ValueError(f"Unsupported image shape: {image.shape}")
    return width, height

def generate_video(
    image_path: str = None,
    positive_prompt: str = "a cute anime girl with massive fennec ears and a big fluffy tail wearing a maid outfit turning around",
    negative_prompt: str = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走 ",
    change_resolution: bool = False,
    width: int = 832,
    height: int = 480,
    seed: int = 82628696717253,
    use_causvid: bool = False,
    causvid_lora_strength: float = 0.25,
    steps: int = 20,
    cfg_scale: float = 1.0,
    sampler_name: str = "uni_pc",
    scheduler: str = "simple",
    frames: int = 33,
    fps: int = 16,
    remove_first_frame: bool = True,
    match_colors: bool = False,
    output_format: str = "mp4",
    overwrite: bool = False,
    use_lora_1: bool = False,
    LoRA_1_Strength: float = 1.0,
    use_lora_2: bool = False,
    LoRA_2_Strength: float = 1.0
):

    with torch.inference_mode():

        # Initialize nodes
        unet_loader = UnetLoaderGGUF()
        model_sampling = ModelSamplingSD3()
        clip_loader = CLIPLoader()
        clip_encode_positive = CLIPTextEncode()
        clip_encode_negative = CLIPTextEncode()
        vae_loader = VAELoader()
        # clip_vision_loader = CLIPVisionLoader()
        # clip_vision_encode = CLIPVisionEncode()
        image_scaler = ImageScale()
        load_image = LoadImage()
        load_lora = LoraLoaderModelOnly()
        load_lora_1 = LoraLoaderModelOnly()
        load_lora_2 = LoraLoaderModelOnly()
        wan_vace_to_video = WanVaceToVideo()
        trim_video_latent = TrimVideoLatent()
        ksampler = KSampler()
        vae_decode = VAEDecode()
        save_webp = SaveAnimatedWEBP()
        save_webm = SaveWEBM()

        print("Loading Text_Encoder...")
        clip = clip_loader.load_clip(text_encoder, "wan", "default")[0]

        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        torch.cuda.empty_cache()
        gc.collect()

        if image_path is None:
            print("Please upload an image file:")
            image_path = upload_image()
        if image_path is None:
            print("No image uploaded!")
        loaded_image = load_image.load_image(image_path)[0]
        # clip_vision = clip_vision_loader.load_clip("clip_vision_h.safetensors")[0]
        # clip_vision_output = clip_vision_encode.encode(clip_vision, loaded_image, "none")[0]

        width_int, height_int = image_width_height(loaded_image)

        if change_resolution:
            print("Changing Image Resolution...")
            loaded_image = image_scaler.upscale(
                loaded_image,
                "lanczos",
                width,
                height,
                "disabled"
            )[0]
        else:
            width = width_int
            height = height_int

        print(f"Image width is {width} and height is {height}")

        # del clip_vision
        # torch.cuda.empty_cache()
        # gc.collect()

        print("Loading VAE...")
        vae = vae_loader.load_vae(vae_model)[0]

        positive_out, negative_out, out_latent, trim_latent = wan_vace_to_video.encode(
            positive, negative, vae, width, height, frames, 1, 1, None, None, loaded_image
        )

        print("Loading Unet Model...")
        model = unet_loader.load_unet(dit_model)[0]

        if use_causvid:
            # if lora is not None:
            print("Loading causvid Lora...")
            model = load_lora.load_lora_model_only(model, causvid_lora, causvid_lora_strength)[0]


        if use_lora_1:
            if lora_1 is not None:
                print("Loading LoRA 1...")
                model = load_lora_1.load_lora_model_only(model, lora_1, LoRA_1_Strength)[0]

        if use_lora_2:
            if lora_2 is not None:
                print("Loading LoRA 2...")
                model = load_lora_2.load_lora_model_only(model, lora_2, LoRA_2_Strength)[0]


        model = model_sampling.patch(model, 8)[0]

        base_name = "output"
        if not overwrite:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            base_name += f"_{timestamp}"

        print("Generating video...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive_out,
            negative=negative_out,
            latent_image=out_latent
        )[0]

        del model
        torch.cuda.empty_cache()
        gc.collect()

        if remove_first_frame:
            print("Trimming video latent...")
            sampled = trim_video_latent.op(sampled, trim_latent)[0]

        # sampled = trim_video_latent.op(sampled, 1)[0]

        try:
            print("Decoding latents...")
            decoded = vae_decode.decode(vae, sampled)[0]

            del vae
            del sampled
            torch.cuda.empty_cache()
            gc.collect()

            if match_colors:
                print("Matching color of video frames to Reference Image...")
                decoded = colormatch(loaded_image, decoded)[0]


            output_path = ""
            if frames == 1:
                print("Single frame detected - saving as PNG image...")
                output_path = save_as_image(decoded[0], base_name)
                # print(f"Image saved as PNG: {output_path}")

                display(IPImage(filename=output_path))
            else:
                if output_format.lower() == "webm":
                    print("Saving as WEBM...")
                    output_path = save_as_webm(
                        decoded,
                        base_name,
                        fps=fps,
                        codec="vp9",
                        quality=10
                    )
                elif output_format.lower() == "mp4":
                    print("Saving as MP4...")
                    output_path = save_as_mp4(decoded, base_name, fps)
                else:
                    raise ValueError(f"Unsupported output format: {output_format}")

                # print(f"Video saved as {output_format.upper()}: {output_path}")

                display_video(output_path)

                global vid_15fps

                vid_15fps = output_path

                del decoded
                torch.cuda.empty_cache()
                gc.collect()

        except Exception as e:
            print(f"Error during decoding/saving: {str(e)}")
            raise
        finally:
            clear_memory()

def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode

    video_data = open(video_path,'rb').read()

    # Determine MIME type based on file extension
    if video_path.lower().endswith('.mp4'):
        mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'):
        mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'):
        mime_type = "image/webp"
    else:
        mime_type = "video/mp4"  # default

    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()

    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="{mime_type}">
    </video>
    """))

print("✅ Environment Setup Complete!")






In [ ]:
# @title 2. Generate a 4 Sec Clip Using Image and Prompt (Audio)

# 1. Imports
import os
import glob
import shutil
import random
import sys
import torch
from google.colab import files

# 2. (Optional) import your generate_video helper
# from your_module import generate_video

# 3. Prepare directories
COMFY_DIR  = "/content/ComfyUI"
INPUT_DIR  = os.path.join(COMFY_DIR, "input")
OUTPUT_DIR = os.path.join(COMFY_DIR, "output")
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 4. Upload initial image
print("▶ Upload your initial image (png/jpg):")
img_up = files.upload()
if not img_up:
    raise RuntimeError("No image uploaded!")
orig_img = next(iter(img_up))
safe_img = "init_image.png"
shutil.copy(orig_img, os.path.join(INPUT_DIR, safe_img))
print(f"✅ Image saved to {INPUT_DIR}/{safe_img}")

your_img    = os.path.join(INPUT_DIR, safe_img)
your_prompt = "a girl dancing"

# 5. Build parameters for generate_video
video_params = {
    "image_path": your_img,
    "positive_prompt": your_prompt,
    "negative_prompt": (
        "bad quality, blurry, messy, chaotic, man, bad anatomy, bad hands, "
        "missing fingers, extra limbs, malformed hands, twisted pose, duplicates, "
        "jpeg artifacts, watermark, cartoon, flat colors"
    ),
    "change_resolution": True,
    "width": 432,
    "height": 768,
    "seed": random.randint(0, 2**32 - 1),
    "use_causvid": True,
    "causvid_lora_strength": 0.8,
    "steps": 4,
    "cfg_scale": 1.0,
    "sampler_name": "uni_pc",
    "scheduler": "simple",
    "frames": 67,
    "fps": 15,
    "remove_first_frame": True,
    "match_colors": True,
    "output_format": "mp4",
    "overwrite": True,
    "use_lora_1": False,
    "LoRA_1_Strength": 1.0,
    "use_lora_2": False,
    "LoRA_2_Strength": 1.0,
}

# 6. Change to COMFY_DIR so that generate_video writes into OUTPUT_DIR there
os.chdir(COMFY_DIR)

# 7. Clear CUDA cache (if using GPU)
if 'torch' in sys.modules:
    torch.cuda.empty_cache()

# 8. Generate the clip
print("▶ Generating video clip…")
generate_video(**video_params)

# 9. Find the newly created clip in OUTPUT_DIR
mp4_files = glob.glob(os.path.join(OUTPUT_DIR, "*.mp4"))
if not mp4_files:
    raise RuntimeError("No .mp4 output was found in OUTPUT_DIR!")
latest = max(mp4_files, key=os.path.getctime)

# 10. Move and rename to final_video.mp4 in COMFY_DIR
final_path = os.path.join(COMFY_DIR, "final_video.mp4")
shutil.move(latest, final_path)

print(f"✅ Final video saved to: {final_path}")


In [ ]:
# @title 2.5. Generate & Stitch Video from Pre-processed Audio
# This notebook assumes you have already run the transcription session and have a `transcription_data.pkl` file.

# 0. Install / Imports
# Note: We are NOT installing whisper or librosa here to save memory.
import os
import glob
import shutil
import time
import random
import pickle
import moviepy.editor as mpy
from google.colab import files
from IPython.display import Video, display

# This assumes the 'google.genai' library is available in your environment.
# If not, you may need to run: !pip install -q google-generativeai
import google.generativeai as genai

# This assumes 'torch' and the 'generate_video' and 'clear_memory' functions are defined elsewhere in your Colab notebook.
# Make sure the cell defining those functions has been run.
import torch

# --- CONFIGURATION ---

# 1. Point to your API key
# IMPORTANT: Replace with your actual Google AI API key
GOOGLE_API_KEY = "your API key"
genai.configure(api_key=GOOGLE_API_KEY)
client = genai.GenerativeModel('gemini-1.5-flash-latest') # Using a slightly more modern name for the client

# 2. Prepare directories
COMFY_DIR = "/content/ComfyUI"
INPUT_DIR = os.path.join(COMFY_DIR, "input")
OUTPUT_DIR = os.path.join(COMFY_DIR, "output")
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- SETUP ---

# 3. Upload initial image
print("▶ Upload your initial image (png/jpg):")
img_up = files.upload()
if not img_up:
    raise RuntimeError("No image uploaded!")
orig_img = next(iter(img_up))
safe_img = "init_image.png"
shutil.copy(orig_img, os.path.join(INPUT_DIR, safe_img))
print(f"✅ Image saved to {INPUT_DIR}/{safe_img}")

# 4. Upload the transcription data file
print("\n▶ Upload your 'transcription_data.pkl' file:")
pkl_up = files.upload()
if not pkl_up:
    raise RuntimeError("No .pkl file uploaded!")
pkl_path = next(iter(pkl_up))
print(f"✅ Loaded {pkl_path}")

# 5. Load segments and split
with open(pkl_path, 'rb') as f:
    transcription_data = pickle.load(f)

segments = transcription_data['segments']
duration = transcription_data['duration']

interval = 4.5  # seconds per clip
num_int = int(duration // interval) + 1
print(f"\n• Audio duration {duration:.1f}s → Will generate {num_int} video segments.")

# Helper function to extract the last frame from a video
def extract_last_frame(video_path, output_path="last_frame.png"):
    """Extracts the very last frame of a video clip."""
    try:
        with mpy.VideoFileClip(video_path) as clip:
            # Save the last frame to the specified output_path
            clip.save_frame(output_path, t=clip.duration - 0.01) # Go to just before the end
        return output_path
    except Exception as e:
        print(f"!! Error extracting last frame: {e}")
        # Fallback to avoid crashing the loop
        return os.path.join(INPUT_DIR, "init_image.png")


# --- VIDEO GENERATION LOOP ---

# 6. Loop and generate each video clip
prev_img = os.path.join(INPUT_DIR, safe_img)
clip_files = []

# Clean memory before starting the loop
if 'torch' in sys.modules:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
if 'clear_memory' in globals():
    clear_memory()

for i in range(num_int):
    start_time, end_time = i * interval, (i + 1) * interval

    # Collect text for the current segment
    texts = [
        seg["text"].strip() for seg in segments
        if seg["start"] < end_time and seg["end"] > start_time
    ]
    if not texts:
        print(f"\n- Skipping segment {i+1}/{num_int} (no text found).")
        continue

    seg_txt = " ".join(texts)

    # Generate a descriptive prompt using the Gemini API
    print(f"\n▶ Generating prompt for segment {i+1}/{num_int}...")
    print(f"  Text: '{seg_txt}'")

    prompt_request = (
        f"""You are a girl is standing, looking into the camera, saying these words: '{seg_txt}' Just describe what you're doing physically while speaking — including hand movements, facial expressions, eye and head movements. Ignore the meaning of the words. Keep it natural, realistic, and focused only on visible actions during speech.i dont want a breakdown li when i am saying this i will do this i just want a short description of the actions under 100 words and start with 'a girl' and instead of saying camera say 'front'"""
    )

    try:
        response = client.generate_content(prompt_request)
        ai_prompt = response.text.strip()
    except Exception as e:
        print(f"  !! Gemini API failed: {e}. Using a fallback prompt.")
        ai_prompt = "a girl talking to the front, moving her hands naturally"

    print(f"  Prompt: {ai_prompt}")

    # It's good practice to change directory right before the call if the helper expects it
    os.chdir(COMFY_DIR)

    # --- Set parameters for generate_video ---
    # These are the same as your original script
    video_params = {
        "image_path": prev_img,
        "positive_prompt": ai_prompt,
        "negative_prompt": """bad quality, blurry, messy, chaotic, man, bad anatomy, bad hands, bad fingers, missing fingers, missing limbs, extra fingers, extra hand, extra limbs, multiple arms, multiple legs,three hands, duplicated limbs, dislocated joints, broken limbs, fused limbs, malformed hands, malformed limbs, wrong limb count, anatomically incorrect, asymmetrical body, twisted pose, unnatural pose, distorted proportions, duplicates, worst quality, low quality, jpeg artifacts, signature, watermark, blurry, bad feet, mutation, deformed, cross-eyed, malformed facial features, lopsided face, lazy eye, misshapen head, half body, cartoon, flat colors, childish drawing, sketch, painting style""",
        "change_resolution": True,
        "width": 432,
        "height": 768,
        "seed": random.randint(0, 2**32 - 1),  # Use a random seed for variation
        "use_causvid": True,
        "causvid_lora_strength": 0.8,
        "steps": 4,
        "cfg_scale": 1.0,
        "sampler_name": "uni_pc",
        "scheduler": "simple",
        "frames": 67,  # 65 + 2
        "fps": 15,
        "remove_first_frame": True,
        "match_colors": True,
        "output_format": "mp4",
        "overwrite": True,
        "use_lora_1": False,
        "LoRA_1_Strength": 1.0,
        "use_lora_2": False,
        "LoRA_2_Strength": 1.0,
    }

    print(f"  Generating clip_{i:02d}.mp4...")

    # Clear CUDA cache before each generation
    if 'torch' in sys.modules:
        torch.cuda.empty_cache()

    # Generate the video clip
    generate_video(**video_params)

    # Find the newly created video file and move it
    output_files = glob.glob(os.path.join(OUTPUT_DIR, "*.mp4"))
    if not output_files:
        raise RuntimeError("Video generation failed! No output file was created.")

    latest_file = max(output_files, key=os.path.getctime)
    clip_name = f"/content/clip_{i:02d}.mp4"
    shutil.move(latest_file, clip_name)
    clip_files.append(clip_name)

    # Display the generated clip inline
    display(Video(clip_name, embed=True, width=320))

    # Prepare for the next iteration by extracting the last frame
    print(f"  Extracting last frame from {clip_name}...")
    prev_img = extract_last_frame(clip_name, output_path=f"/content/last_frame_{i:02d}.png")

# --- FINALIZATION ---

# 7. Stitch all generated clips into a final video
if clip_files:
    print(f"\n🔗 Concatenating {len(clip_files)} clips into final_video.mp4...")

    # Load all clips with moviepy
    video_clips = [mpy.VideoFileClip(p) for p in clip_files]

    # Concatenate them
    final_clip = mpy.concatenate_videoclips(video_clips, method="compose")

    # Write the final video file
    final_clip.write_videofile("final_video.mp4", fps=30, codec="libx264", audio=False)

    print("\n✅ Done! Here’s your final stitched video (without audio):")
    display(Video("final_video.mp4", embed=True, width=480))
else:
    print("\nNo clips were generated. Cannot create a final video.")

In [ ]:
# @title 4.Reshape
import cv2
from IPython.display import Video

# 1️⃣ Resize the video (same as before)
input_path  = '/content/ComfyUI/final_video.mp4'
output_path = '/content/output_video.mp4'

new_width, new_height = 452, 676
cap   = cv2.VideoCapture(input_path)
fps   = cap.get(cv2.CAP_PROP_FPS)
fourcc= cv2.VideoWriter_fourcc(*'mp4v')
out   = cv2.VideoWriter(output_path, fourcc, fps, (new_width, new_height))

while True:
    ret, frame = cap.read()
    if not ret: break
    resized = cv2.resize(frame, (new_width, new_height), interpolation=cv2.INTER_AREA)
    out.write(resized)

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"✅ Saved resized video to {output_path}")

